## Data Preparation
-----

This notebook focuses on generating **nested training datasets** based on total audio duration.

We create progressively larger subsets (e.g., 1h → 2h → 5h → 10h), where each dataset includes all samples from the previous one. This supports consistent and reproducible training experiments.

- Uses a single shuffled manifest  
- Builds subsets using cumulative audio duration  
- Outputs separate CSV manifests for each duration  

> **Note:** Audio files are **not copied**. All subsets reference the same underlying audio paths, making the process fast and storage-efficient.


## 1.Setup

### 1.1 Python Imports

In [22]:
## Enviroment helper variable to help 
# when running the notebook in different environments (e.g. local vs colab)
ENV = "local"  # options: "local", "colab"

In [23]:
import sys
import os
import re
import json
from functools import partial
from dotenv import load_dotenv
from huggingface_hub import login
from pathlib import Path
from functools import partial

import pandas as pd
from evaluate import load
from datasets import DatasetDict, Dataset, Audio, Features, Value
import torch
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer
)

# Add project root (recommended)
if ENV == "jupyter-hub":
    PROJECT_SRC = Path("/content/drive/MyDrive/chichewa-asr/src")   
else:
    PROJECT_SRC = Path().cwd().parents[1]

# Add project src to path to allow imports from src folder
sys.path.append(str(PROJECT_SRC))

# Import functions from src.train.train_whisper
from src.train.train_mms import (
   prepare_mms_batch,
   DataCollatorCTCWithPadding,
    compute_mms_corpus_metrics
)

from src.data_utils.data_utils import (
    load_audio_data, 
    show_random_elements,
    remove_special_characters,
    extract_all_chars
)

from src.train.whisper_duration_experiment import run_evaluation

### 1.2. Input Folder and Other Configuration

In [24]:
# Base data directory
DIR_BASE = Path.cwd().parents[1]
DIR_DATA = DIR_BASE.joinpath('data')

# Direcotory for test data and manifest file
DIR_TEST = DIR_DATA / "test"
FILE_MANIFEST_TEST = DIR_TEST / "metadata.csv"

# Directory for dev data and manifest file
DIR_DEV = DIR_DATA / "dev"
FILE_MANIFEST_DEV = DIR_DEV / "metadata.csv"

# Directory for nested duration based data
DIR_DEV_NESTED_DURATION = DIR_DATA / "dev_nested_duration"

# Hyperparameter config file path
FILE_CONFIG = DIR_BASE/"configs/whisper_params.yaml"

# Outputs directory where we keep the results of the experiments
DIR_OUTPUTS = DIR_BASE / "outputs"
DIR_RESULTS = DIR_OUTPUTS / "duration_exp_whisper_small"
DIR_RESULTS.mkdir(parents=True, exist_ok=True)


# Model checkpoint directory (for saving model checkpoints during training)
# Model checkpoint directory (for saving model checkpoints during training)
DIR_MODELS = DIR_BASE / "models"
DIR_WANDB_LOGS = DIR_MODELS / "wandb"

DIR_MODEL_CHECKPOINTS = DIR_MODELS / "checkpoints"
DIR_MODEL_CHECKPOINTS.mkdir(parents=True, exist_ok=True)


In [25]:
# ===============================
# GLOBAL VARIABLES
# ===============================
# CHARS_TO_REMOVE_REGEX = '[\,\?\.\!\-\;\:\"\“\%\‘\”\�\']'
CHARS_TO_REMOVE_REGEX = re.compile(r'[,?.!\-;:\"""%\"�—…–]|\n')

# Model and processor names
MODEL_NAME = "facebook/mms-1b-all"

### 1.3 Hugging Face Hub Log in

In [26]:
# Log into Hugging Face Hub (optional, required if you want to push the model to the hub)
if ENV == "colab":
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)

else:
    load_dotenv()
    login(token=os.getenv("HF_TOKEN"))


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### 1.4 Device for Training

In [27]:
# Check if CUDA is available and set the device accordingly
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Using device: {device}")

Using device: cpu


## 2. Define Utility Functions

## 2.MMS Training Process 

### 2.1 Prepare Data and Vocabulary 
- Clean the transcripts 
- Create Character level vocubulary list
- Save vocabulary file 

In [28]:
# ===================================
# 1. LOAD TRAIN AND TEST DATASETS 
# ===================================
# Lets use one hour training data for quick testing
file_manifest_1h = DIR_DEV_NESTED_DURATION / "train_1h.csv"
dataset_train = load_audio_data(file_manifest_1h, audio_dir=DIR_DEV)
dataset_test = load_audio_data(FILE_MANIFEST_TEST, audio_dir=DIR_TEST,audio_fname_col="audio_filename", 
                               split_data=False, duration_col="duration_seconds")

Total duration : 1.00 hrs  (278 utterances)
  train       :   244 utterances  |  0.90 hrs  (89.9%)
  validation  :    34 utterances  |  0.10 hrs  (10.1%)
Total duration : 1.68 hrs  (573 utterances)
  all_data   :   573 utterances  |  1.68 hrs (100.0%)


In [29]:
# ===================================
# 2. VIEW DATASET
# ===================================
remove_columns = ['audio', 'duration', 'audio_fname']
show_random_elements(dataset_train['train'].remove_columns(remove_columns), num_examples=10)

,sentence
0,"ngomvetsetsa bwinobwino kuti munati wina aliyense amatha kupita kukamuona eti? Ssikuti ndi a Relief amene amapanga dongosolo amenewo? mmm, ndamva kaya bwinobwino? mmm ndi momwemo."
1,ndiye basi ndiyesa kutani ndiyesa kwa munthu wina eetu\nuh \neetu\nkodi pali nthawi inuyo munapempha thandizo koma munakanizidwa?
2,Wa nsembe uja ndiye ndi uti pamenepa kodi?
3,Ndathira tima eye drops mmasomu ndiye ndikudikirira tiyambe kugwira ntchito; sindikuona apa ndango gona ndizingomutuma Mercy.
4,Nkhani zatha ndiye sindikudziwa kuti ndinene kuti chiyani apa; koma kujambulitsa ndiye kwatheka panopa mwina ndili pafupifupi; ndikuthamangira 380 tinthu tojambula.
5,Kodi nyumba akushala ija ndiyakwawo kapena angopanga rent ku nchenga utuwaku?
6,Chilungamo chake ineyo ndikhodza kusankha ku walkers ferry chifukwa zinthu of course ndizodulirako; koma ndi ku mudzi ndiye sikuti zingadule mmene zikudulira mtauni panopo; panopo moyo wa mtauni ndi ukapolo koopsya kwenikweni Lilongwe; eyatu.
7,Ndimafotokozera Chikondi kuti Kamuzu adanena kuti ufulu umene talandirawu musaganize kuti wangobwera kumwamba uko ngati manna okha phi.
8,amene mukulipiliridwa panopanu tidzakusiyani ngati simunkhonza tizatenga anthu ena inuyo kukusiyani chifukwa choti sitikuona chidwi china chake ndiye ineyo mkati mwanga ndimatha kumadzi
9,Ndikubwera ndi ma prophecies a nyoo! yaaa yes ma prophecies a nyooo yaaa


In [30]:
# ===================================
# 3. PREPROCESS THE DATASET 
# ===================================
dataset_train = dataset_train.map(remove_special_characters)
dataset_test = dataset_test.map(remove_special_characters)


Map: 100%|██████████| 573/573 [00:00<00:00, 61001.48 examples/s]


In [31]:
show_random_elements(dataset_train['train'].remove_columns(remove_columns), num_examples=10)

,sentence
0,nkhanizi zimangothtera pa driver basi kwinako sizipitilira
1,mwapitano ku living waters malandilidwe ake anali otani tsiku loyamba\nukonso anatilandira bwino kwabasi chifukwa chonena kuti anaona kuti mpingonso aonjezeranso nkhosa zina nde anatilandiranso kwabasi ngati mpingo mmmhu
2,ndatsogola mundipeza pa ethiopia pa bomapo
3,ainu ake akatundu mtauni ya lilongwe awauza zoti apente ma buildings awo pofika pa 6 july
4,hello aa ndikupangapo record zinthu ndiye ndipanganso record chakochi capital ili ku area i think 10 area 10 si 12 imene ija ufufuze
5,lo mmm ndithu mwati abusa andani amenewo a chawanda a chawanda ya mwati ndi aku ntaja ku ntaja ndiy e munachoka kunoko kupita ku ntaja kepana nthawi imeneyoyo ndikuti timakhala komweko ku nsana ku ntaja komweko
6,ndi mapemphero apaderadera \nmunandiuzapo kuti muligulu la intersession nde mukamapanga intersession mumatha kuuzako anthu enake oti saligulumo koma ndiapatchalichi pompo kaya ndi
7,basi kagoneni mawa paja ndi la sukulu basi kagoneni mawa paja ndi la sukulu
8,zi mukhoza kukatenga mafuta yanyama mukaphika mukagula nyama ndiye kuti zosezo zikukhala magulu yotii mutengako tisinjiro kuthira kuphala ndiye kuti zakudya zamaguluwo tilikuchita ndithu
9,ndikunyamukatu eee tionana tikumana pompano


In [32]:
# ==================================
# 3. BUILD VOCABULARY
# ==================================
vocab_train = dataset_train['train'].map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=dataset_train['train'].column_names)
vocab_validation = dataset_train['validation'].map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=dataset_train['validation'].column_names)

# Combine the vocabularies from train and validation sets and sort them
vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_validation["vocab"][0]))
vocab_list.sort()

vocab_dict = {v: k for k, v in enumerate(vocab_list)}

print(f"Vocabulary size: {len(vocab_list)}")
print(f"Vocabulary: {vocab_dict}")

# ==================================
# 4. NORMALIZE VOCABULARY
# ==================================
# Replace space with | to avoid confusion with padding token
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

# Add special tokens to the vocabulary
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)
len(vocab_dict)

# =====================================
# 5. CREATE TARGET LANGUAGE VOCAB_DICT
# AND SAVE AS JSON FILE
# =====================================
target_lang = "nya"
new_vocab_dict = {target_lang: vocab_dict}

# vocb file
vocab_file_path = DIR_MODELS / "mms-1b-chichewa" / "vocab.json"
vocab_file_path.parent.mkdir(parents=True, exist_ok=True)

with open(vocab_file_path, 'w') as vocab_file:
    json.dump(new_vocab_dict, vocab_file)


Map: 100%|██████████| 34/34 [00:00<00:00, 28407.64 examples/s]

Vocabulary size: 38
Vocabulary: {'\n': 0, ' ': 1, '0': 2, '1': 3, '2': 4, '3': 5, '4': 6, '5': 7, '6': 8, '7': 9, '8': 10, '9': 11, 'a': 12, 'b': 13, 'c': 14, 'd': 15, 'e': 16, 'f': 17, 'g': 18, 'h': 19, 'i': 20, 'j': 21, 'k': 22, 'l': 23, 'm': 24, 'n': 25, 'o': 26, 'p': 27, 'q': 28, 'r': 29, 's': 30, 't': 31, 'u': 32, 'v': 33, 'w': 34, 'x': 35, 'y': 36, 'z': 37}


### 2.2  Load Tokenizer
We use ```Wav2Vec2CTCTokenizer```

In [33]:
# Load the tokenizer using the vocab file
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(str(vocab_file_path.parent), unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|", target_lang=target_lang)

# Push the tokenizer to Hugging Face Hub
repo_name = "mms-1b-chichewa"
tokenizer.push_to_hub(repo_name)

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/dmatekenya/mms-1b-chichewa/commit/29fb2b5421034f19eb443ed20ffb62fd69149877', commit_message='Upload tokenizer', commit_description='', oid='29fb2b5421034f19eb443ed20ffb62fd69149877', pr_url=None, repo_url=RepoUrl('https://huggingface.co/dmatekenya/mms-1b-chichewa', endpoint='https://huggingface.co', repo_type='model', repo_id='dmatekenya/mms-1b-chichewa'), pr_revision=None, pr_num=None)

### 2.3  Feature Extractor
In Wav2Vec2/XLS-R/MMS ASR models, the feature extractor prepares raw speech waveforms into the format expected by the pretrained model. Unlike traditional ASR systems that use handcrafted features such as MFCCs, these models operate directly on sampled audio signals.

The feature extractor:
- Takes raw audio arrays as input (typically mono 16 kHz waveforms)
- Outputs normalized and padded tensors (`input_values`)
- Optionally returns an `attention_mask` for batched inference/training

Sampling rate is critical because pretrained checkpoints expect audio sampled at the same rate used during pretraining (commonly 16 kHz). Using a different sampling rate changes the signal distribution and can significantly reduce performance.

Typical configuration:
- `feature_size=1` → model consumes raw waveform amplitudes directly
- `sampling_rate=16000`
- `padding_value=0.0`
- `do_normalize=True` → zero-mean/unit-variance normalization
- `return_attention_mask=True` → especially important for XLS-R/MMS batching

In [34]:
# Feature extractor
feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0, do_normalize=True, return_attention_mask=True)

# Processor - combines the feature extractor and tokenizer
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

### 2.3 Prepare Audio Data

The `train_mms.prepare_mms_batch` function transforms raw audio recordings and transcripts into the numerical format required for training Wav2Vec2/XLS-R/MMS ASR models.

The function:
- Extracts the raw audio waveform from `batch["audio"]`
- Applies the processor/feature extractor to generate normalized `input_values`
- Computes and stores the processed audio length as `input_length`
- Converts the transcript (`sentence`) into tokenized integer `labels`

The resulting dataset example contains:
- `input_values` → processed audio input for the model
- `input_length` → length of the processed audio
- `labels` → tokenized transcript targets used during training

In [35]:
# =========================================
# 1. SET SAMPLE RATE TO 16KHZ FOR WAV2VEC2
# =========================================
dataset_train = dataset_train.cast_column("audio", Audio(sampling_rate=16_000))
dataset_test = dataset_test.cast_column("audio", Audio(sampling_rate=16_000))

# =========================================
# 2. PREPARE AUDIO FEATURES FOR TRAINING
# =========================================
dataset_train = dataset_train.map(
    lambda batch: prepare_mms_batch(batch, processor=processor),
    remove_columns=dataset_train["train"].column_names,
)

remove_columns = [c for c in dataset_test.column_names if c != "audio_fname"]
dataset_test = dataset_test.map(
    lambda batch: prepare_mms_batch(batch, processor=processor),
    remove_columns=remove_columns,
)


Map: 100%|██████████| 573/573 [00:07<00:00, 72.50 examples/s]


### 2.3 Setup Trainer

After preprocessing the audio data, the next step is to configure the MMS training pipeline using the Hugging Face `Trainer`.

A key component of the training setup is the data collator, which prepares batches during training. We will use `train_mms.DataCollatorCTCWithPadding`, a custom collator designed for CTC-based speech recognition models such as MMS/Wav2Vec2/XLS-R.

Unlike NLP tasks, speech recognition inputs (`input_values`) are much longer than output labels (`labels`). For example, an audio sequence may contain tens of thousands of waveform values while the corresponding transcript contains only a small number of character tokens. To improve efficiency, the data collator dynamically pads each batch only to the longest sample within that batch rather than padding all samples to the maximum length in the entire dataset.

The collator:
- Separately pads audio inputs and transcript labels
- Uses the processor to correctly handle speech and text modalities
- Replaces padded label tokens with `-100` so they are ignored during loss computation

In addition to the data collator, the training pipeline also includes:
- A `compute_metrics` function for evaluating Word Error Rate (WER)
- Loading and configuring a pretrained MMS checkpoint
- Defining training hyperparameters and optimization settings

After training, the fine-tuned model will be evaluated on the held-out test set to assess transcription performance.

In [36]:
# ===============================================
# 1. LOAD THE DATA COLLATOR FOR CTC WITH PADDING
# ===============================================
# We use a predefined data collator in train_mms.DataCollatorCTCWithPadding
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

In [37]:
# ===============================================
# 2. LOAD THE MODEL FOR CTC
# ===============================================
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    layerdrop=0.0,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True,
)


Loading weights: 100%|██████████| 1096/1096 [00:00<00:00, 29242.36it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/mms-1b-all
Key            | Status   |                                                                                            
---------------+----------+--------------------------------------------------------------------------------------------
lm_head.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([154]) vs model:torch.Size([42])            
lm_head.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([154, 1280]) vs model:torch.Size([42, 1280])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [38]:
# ===============================================
# 3. CONFIGURE ADAPTER LAYERS
# ===============================================
# Re-initialize the MMS adapter layers so they can be trained specifically
# for the target language and custom vocabulary. While it is possible to
# continue training existing adapter weights via `load_adapter(...)`,
# the pretrained vocabulary and language-specific adapters may not align
# well with the current training data. Re-initializing the adapter layers
# provides a cleaner starting point for fine-tuning on the custom dataset.
model.init_adapter_layers()

# Freeze the base model parameters and only train the adapter layers
model.freeze_base_model()

adapter_weights = model._get_adapters()
for param in adapter_weights.values():
    param.requires_grad = True

In [46]:
# ===============================================
# 4. CONFIGURE TRAINING ARGUMENTS
# ===============================================
training_args = TrainingArguments(
  output_dir=repo_name,
  train_sampling_strategy="group_by_length",
  length_column_name="input_length",
  per_device_train_batch_size=32,
  eval_strategy="steps",
  num_train_epochs=4,
  gradient_checkpointing=True,
  fp16=True,
  save_steps=200,
  eval_steps=100,
  max_steps=100,
  logging_steps=100,
  learning_rate=1e-3,
  warmup_steps=100,
  save_total_limit=2,
  push_to_hub=True,
)
# ===============================================
# 4. INITIALIZE THE TRAINER
# ===============================================
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=partial(compute_mms_corpus_metrics, processor=processor, training_mode=True),
    train_dataset=dataset_train['train'],
    eval_dataset=dataset_train['validation'],
    processing_class=processor.feature_extractor,
)


### 2.4 Run Training

In [ ]:
trainer.train()

/Users/dmatekenya/git-repos/chichewa-asr/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
df = run_evaluation(
    model,
    processor,
    dataset_test,
    duration_label="duration_seconds",
    results_dir=DIR_RESULTS,
    debug=True)

In [ ]:
wer = df['wer']
cer = df['cer']
df_results = df['predictions']

In [ ]:
dataset_test_sample = dataset_test.shuffle(seed=42).select(range(20))
df_results = evaluate_holdout_set(model, processor, dataset_test_sample, text_column="sentence", fname_column="audio_fname")
 

In [ ]:
df_predictions = df_results["predictions"]
cer = df_results["cer"]
wer = df_results["wer"]

In [ ]:
df_results